In [0]:
# Carregamento dos dados - ETAPA BRONZE 
# Os dados foram disponibilizados em arquivos CSV e estão armazenados no volume `olist_raw`. A etapa Bronze será usada para manter esses dados próximos da estrutura original, antes dos tratamentos que serão realizados na etapa Silver.

In [0]:
# Caminho onde os arquivos originais foram armazenados
raw_path = "/Volumes/workspace/default/olist_raw/"

# Verifica quais arquivos estão disponíveis 
arquivos = dbutils.fs.ls(raw_path)

# Apresenta o resultado
display(arquivos)

In [0]:
# Leitura do arquivo de pedidos
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(raw_path + "olist_orders_dataset.csv")
)

# Mostra as primeiras dez linhas
display(orders_df.limit(10))

In [0]:
# Conferindo os tipos de dados
orders_df.printSchema()

In [0]:
# Arquivos carregados
arquivos_bronze = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Leitura dos arquivos e criação de tabelas iniciais
for nome_tabela, arquivo in arquivos_bronze.items():
    
    caminho_arquivo = raw_path + arquivo
    
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(caminho_arquivo)
    )
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.default.bronze_{nome_tabela}")
    )
    
    print(f"Tabela bronze_{nome_tabela} criada com {df.count()} registros.")

In [0]:
# Tabelas criadas na camada Bronze
tabelas_bronze = [
    "bronze_customers",
    "bronze_geolocation",
    "bronze_order_items",
    "bronze_order_payments",
    "bronze_order_reviews",
    "bronze_orders",
    "bronze_products",
    "bronze_sellers",
    "bronze_category_translation"
]

# Conferindo a quantidade de registros das tabelas
for tabela in tabelas_bronze:
    quantidade = spark.table(f"workspace.default.{tabela}").count()
    print(f"{tabela}: {quantidade:,} registros")

In [0]:
# Levantando as informações das tabelas paa ocatálogo de dados

for tabela in tabelas_bronze:
    
    df = spark.table(f"workspace.default.{tabela}")
    
    print(f"\n{'=' * 70}")
    print(f"TABELA: {tabela}")
    print(f"REGISTROS: {df.count():,}")
    print(f"COLUNAS: {len(df.columns)}")
    print(f"{'=' * 70}")
    
    df.printSchema()